In [ ]:
import os, json, shutil, glob
from collections import defaultdict
import yaml
from ultralytics import YOLO


SRC_BASE = "/kaggle/input/datasets/minhkhanh17/seadronessee/compressed"
WORK_BASE = "/kaggle/working/seadronessee/compressed"

# copy dataset sang working để có thể ghi labels/json
if not os.path.exists(WORK_BASE):
    os.makedirs("/kaggle/working/seadronessee", exist_ok=True)
    shutil.copytree(SRC_BASE, WORK_BASE)

base = WORK_BASE
print("✅ Using base:", base)

✅ Using base: /kaggle/working/seadronessee/compressed


In [ ]:
# 1) COCO -> YOLO converter
def coco_to_yolo(coco_json_path, images_dir, output_labels_dir, write_empty=False):
    os.makedirs(output_labels_dir, exist_ok=True)

    with open(coco_json_path, "r") as f:
        coco = json.load(f)

    categories = coco.get("categories", [])
    cat_id_to_yolo = {cat["id"]: i for i, cat in enumerate(sorted(categories, key=lambda x: x["id"]))}

    images = {img["id"]: (img["file_name"], img["width"], img["height"]) for img in coco.get("images", [])}

    anns_by_image = defaultdict(list)
    for ann in coco.get("annotations", []):
        if ann.get("iscrowd", 0) == 1:
            continue
        if "bbox" not in ann or ann["bbox"] is None:
            continue
        anns_by_image[ann["image_id"]].append(ann)

    missing_images = 0
    written = 0
    skipped_invalid_box = 0

    for image_id, (file_name, w, h) in images.items():
        if not w or not h:
            continue

        img_path = os.path.join(images_dir, file_name)
        if not os.path.exists(img_path):
            img_path2 = os.path.join(images_dir, os.path.basename(file_name))
            if os.path.exists(img_path2):
                file_name = os.path.basename(file_name)
            else:
                missing_images += 1
                continue

        yolo_lines = []
        for ann in anns_by_image.get(image_id, []):
            cat_id = ann.get("category_id", None)
            if cat_id not in cat_id_to_yolo:
                continue

            x, y, bw, bh = ann["bbox"]

            if bw is None or bh is None or bw <= 0 or bh <= 0:
                skipped_invalid_box += 1
                continue

            cls = cat_id_to_yolo[cat_id]

            x_center = (x + bw / 2) / w
            y_center = (y + bh / 2) / h
            bw_norm = bw / w
            bh_norm = bh / h

            # clamp [0,1]
            x_center = min(max(x_center, 0.0), 1.0)
            y_center = min(max(y_center, 0.0), 1.0)
            bw_norm = min(max(bw_norm, 0.0), 1.0)
            bh_norm = min(max(bh_norm, 0.0), 1.0)

            if bw_norm <= 0 or bh_norm <= 0:
                skipped_invalid_box += 1
                continue

            yolo_lines.append(f"{cls} {x_center:.6f} {y_center:.6f} {bw_norm:.6f} {bh_norm:.6f}")

        stem = os.path.splitext(os.path.basename(file_name))[0]
        out_txt = os.path.join(output_labels_dir, stem + ".txt")

        if yolo_lines:
            with open(out_txt, "w") as f:
                f.write("\n".join(yolo_lines))
            written += 1
        else:
            if write_empty:
                open(out_txt, "w").close()
                written += 1

    print(f"Wrote labels for {written} images to: {output_labels_dir}")
    if missing_images:
        print(f"Skipped {missing_images} images (not found in {images_dir})")
    if skipped_invalid_box:
        print(f"Skipped {skipped_invalid_box} invalid bboxes (bw/bh <= 0)")

In [ ]:
# 2) Filter COCO to ONE class
def filter_coco_one_class(coco_in_path, coco_out_path, keep_name="swimmer"):
    with open(coco_in_path, "r") as f:
        coco = json.load(f)

    # find category by name
    cats = coco.get("categories", [])
    keep_cats = [c for c in cats if c.get("name") == keep_name]
    if not keep_cats:
        raise ValueError(
            f"Category '{keep_name}' not found. Available: {[c.get('name') for c in cats]}"
        )
    keep_cat_id = keep_cats[0]["id"]

    # keep anns
    kept_anns = [
        a for a in coco.get("annotations", [])
        if a.get("iscrowd", 0) == 0 and a.get("category_id") == keep_cat_id and a.get("bbox") is not None
    ]

    # keep images referenced
    keep_image_ids = set(a["image_id"] for a in kept_anns)
    kept_images = [img for img in coco.get("images", []) if img["id"] in keep_image_ids]

    # remap category to single class id=0
    for a in kept_anns:
        a["category_id"] = 0

    coco_out = {
        **coco,
        "images": kept_images,
        "annotations": kept_anns,
        "categories": [{"id": 0, "name": keep_name}],
    }

    os.makedirs(os.path.dirname(coco_out_path), exist_ok=True)
    with open(coco_out_path, "w") as f:
        json.dump(coco_out, f)

    print(f"Saved filtered COCO: {coco_out_path}")
    print(f"Images kept: {len(kept_images)} | Annotations kept: {len(kept_anns)}")

In [ ]:
# 3) Build paths
train_json = os.path.join(base, "annotations", "instances_train.json")
val_json   = os.path.join(base, "annotations", "instances_val.json")

train_images = os.path.join(base, "images", "train")
val_images   = os.path.join(base, "images", "val")

train_json_sw = os.path.join(base, "annotations", "instances_train_swimmer.json")
val_json_sw   = os.path.join(base, "annotations", "instances_val_swimmer.json")

# 4) Filter + Convert
filter_coco_one_class(train_json, train_json_sw, keep_name="swimmer")
filter_coco_one_class(val_json,   val_json_sw,   keep_name="swimmer")

labels_sw_train = os.path.join(base, "labels_swimmer", "train")
labels_sw_val   = os.path.join(base, "labels_swimmer", "val")

coco_to_yolo(train_json_sw, train_images, labels_sw_train)
coco_to_yolo(val_json_sw,   val_images,   labels_sw_val)

# 5) (Optional) Rename labels_swimmer -> labels + clear cache
old_dir = os.path.join(base, "labels_swimmer")
new_dir = os.path.join(base, "labels")

if os.path.exists(new_dir):
    shutil.rmtree(new_dir)
shutil.move(old_dir, new_dir)

# clear caches
for p in glob.glob(os.path.join(new_dir, "*.cache")):
    os.remove(p)
for p in glob.glob(os.path.join(new_dir, "**", "*.cache"), recursive=True):
    os.remove(p)

print("✅ Renamed labels_swimmer → labels and cleared caches")

✅ Saved filtered COCO: /kaggle/working/seadronessee/compressed/annotations/instances_train_swimmer.json
   Images kept: 6898 | Annotations kept: 37096
✅ Saved filtered COCO: /kaggle/working/seadronessee/compressed/annotations/instances_val_swimmer.json
   Images kept: 1286 | Annotations kept: 6206
✅ Wrote labels for 6898 images to: /kaggle/working/seadronessee/compressed/labels_swimmer/train
✅ Wrote labels for 1286 images to: /kaggle/working/seadronessee/compressed/labels_swimmer/val
✅ Renamed labels_swimmer → labels and cleared caches


In [ ]:
# 6) Create YOLO YAML (Kaggle)
yaml_obj = {
    "path": base,
    "train": "images/train",
    "val": "images/val",
    "nc": 1,
    "names": ["swimmer"],
}

yaml_path = "/kaggle/working/seadronessee_swimmer.yaml"
with open(yaml_path, "w") as f:
    yaml.safe_dump(yaml_obj, f, sort_keys=False, allow_unicode=True)

print("✅ saved:", yaml_path)
print(yaml.safe_dump(yaml_obj, sort_keys=False, allow_unicode=True))

✅ saved: /kaggle/working/seadronessee_swimmer.yaml
path: /kaggle/working/seadronessee/compressed
train: images/train
val: images/val
nc: 1
names:
- swimmer



In [ ]:
DATA_YAML = "/kaggle/working/seadronessee_swimmer.yaml"
RUNS_DIR  = "/kaggle/working/runs_people"
os.makedirs(RUNS_DIR, exist_ok=True)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [10]:
tune_model = YOLO("yolov8n.pt")

tune_model.tune(
    data=DATA_YAML,
    epochs=10,
    iterations=10,
    imgsz=640,
    batch=16,
    device=0,
    project=RUNS_DIR,
    name="yolov8n_tune",
    single_cls=True,
)

Tuner: Initialized Tuner instance with 'tune_dir=/kaggle/working/runs_people/yolov8n_tune'
Tuner: 💡 Learn about tuning at https://docs.ultralytics.com/guides/hyperparameter-tuning
Tuner: Starting iteration 1/10 with hyperparameters: {'lr0': 0.01, 'lrf': 0.01, 'momentum': 0.937, 'weight_decay': 0.0005, 'warmup_epochs': 3.0, 'warmup_momentum': 0.8, 'box': 7.5, 'cls': 0.5, 'dfl': 1.5, 'hsv_h': 0.015, 'hsv_s': 0.7, 'hsv_v': 0.4, 'degrees': 0.0, 'translate': 0.1, 'scale': 0.5, 'shear': 0.0, 'perspective': 0.0, 'flipud': 0.0, 'fliplr': 0.5, 'bgr': 0.0, 'mosaic': 1.0, 'mixup': 0.0, 'cutmix': 0.0, 'copy_paste': 0.0, 'close_mosaic': 10}
Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, dat

In [ ]:
def load_best_hp(runs_dir, prefix="yolov8n_tune"):
    cands = sorted([d for d in os.listdir(runs_dir) if d.startswith(prefix)])
    if not cands:
        return {}, None

    tune_dir = os.path.join(runs_dir, cands[-1])  # folder mới nhất theo sort
    best_hp_path = os.path.join(tune_dir, "best_hyperparameters.yaml")

    if not os.path.exists(best_hp_path):
        return {}, tune_dir

    with open(best_hp_path, "r") as f:
        best_hp = yaml.safe_load(f) or {}
    return best_hp, tune_dir

best_hp, tune_dir = load_best_hp(RUNS_DIR)

print("tune_dir =", tune_dir)
print("best_hp keys =", list(best_hp.keys())[:20])
print("best_hp =", best_hp)

tune_dir = /kaggle/working/runs_people/yolov8n_tune
best_hp keys = ['lr0', 'lrf', 'momentum', 'weight_decay', 'warmup_epochs', 'warmup_momentum', 'box', 'cls', 'dfl', 'hsv_h', 'hsv_s', 'hsv_v', 'degrees', 'translate', 'scale', 'shear', 'perspective', 'flipud', 'fliplr', 'bgr']
best_hp = {'lr0': 0.00606, 'lrf': 0.01607, 'momentum': 0.97303, 'weight_decay': 6e-05, 'warmup_epochs': 1.92652, 'warmup_momentum': 0.73741, 'box': 8.23301, 'cls': 0.69572, 'dfl': 2.1128, 'hsv_h': 0.00814, 'hsv_s': 0.78882, 'hsv_v': 0.49261, 'degrees': 0.00087, 'translate': 0.12864, 'scale': 0.50557, 'shear': 0.0114, 'perspective': 4e-05, 'flipud': 0.01777, 'fliplr': 0.327, 'bgr': 0.00048, 'mosaic': 0.85629, 'mixup': 0.01098, 'cutmix': 0.00684, 'copy_paste': 0.02327, 'close_mosaic': 10}


In [ ]:
model = YOLO("yolov8n.pt")

train_kwargs = dict(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project=RUNS_DIR,
    name="yolov8n_final",
    single_cls=True,
    patience=15,
    cache=False,
)


if isinstance(best_hp, dict) and len(best_hp) > 0:
    train_kwargs.update(best_hp)

results = model.train(**train_kwargs)
print("✅ Training done")

Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.00048, box=8.23301, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.69572, compile=False, conf=None, copy_paste=0.02327, copy_paste_mode=flip, cos_lr=False, cutmix=0.00684, data=/kaggle/working/seadronessee_swimmer.yaml, degrees=0.00087, deterministic=True, device=0, dfl=2.1128, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.327, flipud=0.01777, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.00814, hsv_s=0.78882, hsv_v=0.49261, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.00606, lrf=0.01607, mask_ratio=4, max_det=300, mixup=0.01098, mode=train, model=yolov8n.pt, momentum=0.97303, mosaic=0.85629, multi_scale=0.0, name=yolov8n_final, nbs=64, nms=False, opset=None, o

In [ ]:
weight_files = sorted(glob.glob(os.path.join(RUNS_DIR, "**", "weights", "*.pt"), recursive=True))
print("Found weights:", weight_files[-5:])

Found weights: ['/kaggle/working/runs_people/train4/weights/last.pt', '/kaggle/working/runs_people/yolov8n_final/weights/best.pt', '/kaggle/working/runs_people/yolov8n_final/weights/last.pt', '/kaggle/working/runs_people/yolov8n_tune/weights/best.pt', '/kaggle/working/runs_people/yolov8n_tune/weights/last.pt']


In [ ]:
# 0) Paths (Kaggle)
PROJECT_PATH = "/kaggle/working"
RUNS_DIR = os.path.join(PROJECT_PATH, "runs_people")
BACKBONE_DIR = os.path.join(PROJECT_PATH, "backbones")
os.makedirs(BACKBONE_DIR, exist_ok=True)

# Tìm best.pt mới nhất trong runs
best_candidates = sorted(glob.glob(os.path.join(RUNS_DIR, "**/weights/best.pt"), recursive=True))
if not best_candidates:
    raise FileNotFoundError(f"Không tìm thấy best.pt trong {RUNS_DIR}. Hãy train xong trước.")
best_weights = best_candidates[-1]
print("✅ Using best weights:", best_weights)

# Load best model
yolo_best = YOLO(best_weights)

# Lấy metrics (tùy chọn): đọc results.csv mới nhất
yolo_metrics = {}
csv_candidates = sorted(glob.glob(os.path.join(RUNS_DIR, "**/results.csv"), recursive=True))
if csv_candidates:
    df = pd.read_csv(csv_candidates[-1])
    last = df.iloc[-1].to_dict()
    # giữ vài field phổ biến (nếu có)
    for k in ["metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]:
        if k in last:
            yolo_metrics[k] = float(last[k])
print("📌 yolo_metrics:", yolo_metrics)

✅ Using best weights: /kaggle/working/runs_people/yolov8n_tune/weights/best.pt
📌 yolo_metrics: {'metrics/precision(B)': 0.80206, 'metrics/recall(B)': 0.728, 'metrics/mAP50(B)': 0.7243, 'metrics/mAP50-95(B)': 0.29399}
